
# From Means to Neural Networks: A Companion Notebook

This notebook is the hands-on companion to the "From Means to Neural Networks" module.
It contains **every code sample from the module**, in order, with extra explanation aimed
at programmers who are new to the Python data science stack. If you already know how to
`pip install` a package and write a `for` loop, you can skim the boxed explanations and
go straight to the code.

**How to use this notebook:** run the cells from top to bottom, in order. Later sections
reuse variables (`x`, `y`, `rng`, etc.) defined in earlier ones, exactly the way the
original module's examples build on each other. If you jump around and get a
`NameError`, scroll up and re-run the cell that defines the missing variable.

---

## Setting up your environment

Before any of this code will run, you need Python itself installed (any recent Python 3.9+
works fine) and a handful of third-party packages. "Third-party" just means these packages
aren't part of the Python standard library — they're published on the **Python Package
Index (PyPI)**, the central repository of installable Python packages, and you install them
with **pip**, the package manager that ships with Python.

Open a terminal and run:

```bash
pip install numpy scipy scikit-learn matplotlib pandas jupyter
```

A quick breakdown of what each package does (you'll meet all of them properly as you
work through this notebook):

- **numpy** — fast arrays and linear algebra. The foundation everything else is built on.
- **scipy** — additional scientific/statistical routines built on top of numpy.
- **scikit-learn** (imported as `sklearn`) — ready-made machine learning models and tools.
- **matplotlib** — plotting and charts.
- **pandas** — labeled, tabular data (think "spreadsheet in Python").
- **jupyter** — the notebook environment itself (the thing you're reading this in).

If you're running this inside an existing Jupyter notebook and a package is missing, you
can also install it from directly inside a cell by prefixing the command with `!`, e.g.
`!pip install scipy`. That runs the command in your shell rather than as Python code.

One more note on style: throughout this notebook you'll see `import numpy as np`. The
`as np` part is just a nickname — it lets you type `np.mean(...)` instead of the longer
`numpy.mean(...)`. These abbreviations (`np` for numpy, `pd` for pandas, `plt` for
`matplotlib.pyplot`) are so universal in the Python data science world that using anything
else would confuse other people reading your code.



## Section 1 — Describing Data

### Mean and variance

Our first import: `numpy`, almost always nicknamed `np`. Numpy's headline feature is the
**array** — think of it as a Python list that's specialized for fast, whole-collection math.
`np.array([...])` turns an ordinary Python list into a numpy array; from then on, functions
like `np.mean()` and `np.var()` work on the entire array at once, with no manual loop.

- `np.mean(x)` — the average of every value in `x`.
- `np.var(x)` — the population variance (see the note right after this cell about the
  `ddof` argument — it matters).
- `np.sqrt(...)` — square root; here used to turn variance into standard deviation.

`f"..."` strings (called f-strings) let you drop a variable straight into text, e.g.
`f"mean = {mean_x:.2f}"` prints `mean_x` rounded to 2 decimal places.


In [ ]:
import numpy as np

x = np.array([12, 15, 14, 10, 18, 22, 9, 17])

mean_x = np.mean(x)
var_x = np.var(x)  # population variance (divides by n)

print(f"mean = {mean_x:.2f}")
print(f"variance = {var_x:.2f}")
print(f"standard deviation = {np.sqrt(var_x):.2f}")



**A gotcha worth knowing right away:** `np.var()` divides by `n` by default (the
"population" variance). Many stats textbooks, and pandas, divide by `n - 1` instead (the
"sample" variance, a small correction for the fact that you estimated the mean from the
same data you're measuring spread in). Pass `ddof=1` to `np.var()` if you want that
correction: `np.var(x, ddof=1)`. It won't change any conclusions in this module, but it
will produce a slightly different number than pandas' `.var()` if you don't match them up —
worth remembering so it doesn't cost you time during a lab.

### Covariance

`np.cov()` computes how two variables move together. It returns a full 2x2 matrix, not a
single number — indexing it with `[0, 1]` pulls out the one number we actually want.


In [ ]:
x = np.array([1, 2, 3, 4, 5, 6])
y = np.array([2, 4, 5, 4, 5, 8])

cov_xy = np.cov(x, y, ddof=0)[0, 1]
print(f"covariance(x, y) = {cov_xy:.2f}")



`np.cov(x, y, ddof=0)` returns:

```
[[ var(x)       cov(x, y) ]
 [ cov(x, y)    var(y)    ]]
```

The diagonal holds each variable's own variance; the off-diagonal (equal on both sides)
holds the covariance between them. `ddof=0` tells numpy to use the population version
(divide by `n`), matching `np.var()`'s default from the previous cell.

### Correlation

`np.corrcoef()` works the same way — it also returns a matrix, and `[0, 1]` pulls out the
correlation between `x` and `y`, a number always between -1 and 1.


In [ ]:
corr_xy = np.corrcoef(x, y)[0, 1]
print(f"correlation(x, y) = {corr_xy:.3f}")



### Looking at your data, not just summary numbers

`pandas` (nicknamed `pd`) is the standard library for tabular data in Python — think of its
core object, the `DataFrame`, as a spreadsheet living inside your code. This is our first
use of it. `pd.DataFrame({...})` builds a table from a dictionary where each key becomes a
column name and each value is that column's data. `.describe()` then prints count, mean,
standard deviation, min, max, and quartiles for every numeric column in one call — a fast
first look at any new data set.


In [ ]:
import pandas as pd

df = pd.DataFrame({"x": x, "y": y})
print(df.describe())



### Standardizing data (z-scores)

Subtract the mean, divide by the standard deviation. The result always has mean 0 and
standard deviation 1, which makes it possible to compare variables that were originally on
completely different scales (dollars vs. years, for instance).


In [ ]:
z = (x - np.mean(x)) / np.std(x)
print(f"standardized mean = {np.mean(z):.4f}, standardized std = {np.std(z):.4f}")



### Outliers and robustness

`np.append(x, 500)` returns a new array with `500` tacked on the end (numpy arrays are a
fixed size, so "appending" always makes a new array rather than growing the old one in
place — a subtle but important difference from Python lists).


In [ ]:
x_with_outlier = np.append(x, 500)   # one extreme value added
print(f"mean without outlier: {np.mean(x):.2f}, with outlier: {np.mean(x_with_outlier):.2f}")
print(f"std without outlier: {np.std(x):.2f}, with outlier: {np.std(x_with_outlier):.2f}")



### Describing several variables at once

One last new tool here: `np.random.default_rng(seed)` creates a **random number
generator** object, conventionally called `rng`. Passing a fixed `seed` (any integer) means
every call to `rng.uniform(...)` or `rng.normal(...)` produces the *same* "random" numbers
every time you run the notebook — this is what makes an example reproducible. The module
introduces this properly in Section 2; we create `rng` a little early here so this cell
runs on its own.

- `rng.uniform(low, high, size)` draws `size` numbers uniformly between `low` and `high`.
- `rng.normal(mean, std, size)` draws `size` numbers from a normal (bell-curve)
  distribution.
- `df.corr()` computes the correlation between every pair of columns in a DataFrame at
  once — a correlation matrix.


In [ ]:
import pandas as pd

rng = np.random.default_rng(7)  # created here so this example is self-contained

df = pd.DataFrame({
    "hours_studied": rng.uniform(0, 10, 50),
    "sleep_hours": rng.uniform(4, 9, 50),
})
df["exam_score"] = 5 * df["hours_studied"] + 2 * df["sleep_hours"] + rng.normal(0, 5, 50)

print(df.corr())



## Section 2 — Fitting Lines

We regenerate `rng` here with a fixed seed (`42`) so this section's examples are
reproducible independent of Section 1's random numbers above.

- `rng.uniform(0, 10, n)` — `n` random x-values between 0 and 10.
- `rng.normal(0, 4.0, n)` — `n` random noise values, added to a "true" linear relationship
  so the data looks like real, noisy measurements rather than a perfect line.


In [ ]:
import numpy as np

rng = np.random.default_rng(42)
n = 60
x = rng.uniform(0, 10, n)
y = 2.5 * x + 3.0 + rng.normal(0, 4.0, n)   # true slope 2.5, true intercept 3.0

beta1_hat = np.cov(x, y, ddof=0)[0, 1] / np.var(x)
beta0_hat = np.mean(y) - beta1_hat * np.mean(x)

print(f"fitted slope (beta1) = {beta1_hat:.3f}")
print(f"fitted intercept (beta0) = {beta0_hat:.3f}")



### Checking the by-hand formula against a library

`scipy` is a scientific computing library built on top of numpy. We only need one function
from it here: `scipy.stats.linregress`, which fits a line for you and returns an object
with `.slope` and `.intercept` attributes (plus more, like the correlation coefficient).
If scipy isn't installed yet: `pip install scipy`.


In [ ]:
from scipy import stats

result = stats.linregress(x, y)
print(f"scipy slope = {result.slope:.3f}, intercept = {result.intercept:.3f}")



### The matrix-form solution

`np.column_stack([...])` glues 1-D arrays together as side-by-side columns of a single 2-D
array — here, a column of all `1`s next to the `x` column, forming the "design matrix" `X`.
`X.T` is the transpose (rows and columns swapped), `@` is matrix multiplication (numpy's
dedicated operator for it, distinct from `*` which multiplies element-by-element), and
`np.linalg.inv(...)` computes a matrix inverse. Together, `np.linalg.inv(X.T @ X) @ X.T @ y`
is the "normal equations" formula solved directly.


In [ ]:
X = np.column_stack([np.ones(n), x])   # design matrix: intercept column + x column
beta_matrix = np.linalg.inv(X.T @ X) @ X.T @ y
print(f"matrix-form solution: {beta_matrix}")



### Multiple regression: just add another column

Predicting `y` from two inputs (`x` and `x2`) instead of one requires no new machinery —
just one more column in `X`.


In [ ]:
x2 = 0.5 * x + rng.normal(0, 2, n)
y_multi = 2.5 * x - 1.2 * x2 + 3.0 + rng.normal(0, 4.0, n)

X_multi = np.column_stack([np.ones(n), x, x2])
beta_multi = np.linalg.inv(X_multi.T @ X_multi) @ X_multi.T @ y_multi
print(f"multiple regression coefficients: {beta_multi}")



### Doing the same thing with scikit-learn

`scikit-learn` (imported as `sklearn`) is the standard machine-learning library in Python.
Its models all share one interface: create the model object, call `.fit(X, y)` on your
data, then read off results (here, `.coef_` and `.intercept_`). Note that scikit-learn adds
its own intercept column automatically, so we pass it only the `x` and `x2` columns
(`X_multi[:, 1:]` — every row, every column *except* the first one).
If sklearn isn't installed: `pip install scikit-learn`.


In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression().fit(X_multi[:, 1:], y_multi)  # sklearn adds its own intercept
print(f"sklearn coefficients: {model.coef_}, intercept: {model.intercept_}")



### Curves are still "linear" models

Adding a column for `x**2` lets the same machinery fit a parabola. "Linear" refers to being
linear in the *parameters*, not to the shape of the resulting curve.


In [ ]:
X_quadratic = np.column_stack([np.ones(n), x, x**2])
beta_quadratic = np.linalg.inv(X_quadratic.T @ X_quadratic) @ X_quadratic.T @ y
print(f"quadratic fit coefficients: {beta_quadratic}")



### A small worked example, entirely by hand

Five data points, so you can see every number involved without a wall of output.


In [ ]:
x_small = np.array([1, 2, 3, 4, 5])
y_small = np.array([2.1, 3.9, 6.2, 7.8, 10.1])

beta1 = np.cov(x_small, y_small, ddof=0)[0, 1] / np.var(x_small)
beta0 = np.mean(y_small) - beta1 * np.mean(x_small)
print(f"fitted line: y = {beta0:.2f} + {beta1:.2f} * x")

for xi, yi in zip(x_small, y_small):
    prediction = beta0 + beta1 * xi
    print(f"x={xi}: actual y={yi:.1f}, predicted y={prediction:.2f}, residual={yi - prediction:.2f}")



## Section 3 — Measuring Error

We rebuild the same fitted line from Section 2 (same seed, same data) so this section
stands on its own, then compute its residuals — the leftover error at each point,
`actual - predicted`.


In [ ]:
import numpy as np

rng = np.random.default_rng(42)
n = 60
x = rng.uniform(0, 10, n)
y = 2.5 * x + 3.0 + rng.normal(0, 4.0, n)

beta1_hat = np.cov(x, y, ddof=0)[0, 1] / np.var(x)
beta0_hat = np.mean(y) - beta1_hat * np.mean(x)
y_hat = beta0_hat + beta1_hat * x

residuals = y - y_hat
print(f"mean residual = {np.mean(residuals):.4f}")   # should be ~0 for OLS, always
print(f"residual std dev = {np.std(residuals):.3f}")



### Mean squared error (MSE) and R-squared

`sklearn.metrics` is scikit-learn's collection of ready-made scoring functions — no need to
hand-write the averaging/squaring yourself.


In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

mse = mean_squared_error(y, y_hat)
print(f"MSE = {mse:.3f}")


In [ ]:
r2 = r2_score(y, y_hat)
print(f"R^2 = {r2:.3f}")



### Mean absolute error (MAE)

Same idea as MSE, but averaging the absolute value of each residual instead of its square —
so one huge outlier error doesn't dominate the score the way it does under MSE.


In [ ]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y, y_hat)
print(f"MAE = {mae:.3f}  (compare to MSE = {mse:.3f} and RMSE = {np.sqrt(mse):.3f})")



### Comparing two candidate models side by side

`x.reshape(-1, 1)` turns a 1-D array into a 2-D array with one column — scikit-learn
models always expect their input `X` to be 2-D (rows = samples, columns = features), even
when there's only one feature. `-1` tells numpy "figure out this dimension automatically."


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

X_linear = x.reshape(-1, 1)
X_quad = np.column_stack([x, x**2])

for name, X_features in [("linear", X_linear), ("quadratic", X_quad)]:
    m = LinearRegression().fit(X_features, y)
    preds = m.predict(X_features)
    print(
        f"{name:10s}  MSE={mean_squared_error(y, preds):7.2f}  "
        f"MAE={mean_absolute_error(y, preds):6.2f}  "
        f"R2={r2_score(y, preds):.3f}"
    )



## Section 4 — Optimizing Parameters

### Train/test splits, and a first look at overfitting

A batch of new imports, all from scikit-learn:

- `train_test_split` — randomly splits your data into a training portion and a held-out
  test portion (here, 70% / 30%, via `test_size=0.3`). `random_state=0` is scikit-learn's
  version of a random seed — fix it and the split is reproducible.
- `PolynomialFeatures(degree)` — automatically builds the `x`, `x**2`, `x**3`, ... columns
  for you, so you don't have to type them out by hand.
- `make_pipeline(...)` — chains preprocessing steps and a model together into one object,
  so `.fit()` and `.predict()` run the whole chain in one call.


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(1)
n = 40
x = np.linspace(-3, 3, n)
y = 0.5 * x**3 - 2 * x**2 + x + 5 + rng.normal(0, 6, n)

X_train, X_test, y_train, y_test = train_test_split(
    x.reshape(-1, 1), y, test_size=0.3, random_state=0
)

for degree in [1, 3, 11]:
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(X_train, y_train)
    train_mse = mean_squared_error(y_train, model.predict(X_train))
    test_mse = mean_squared_error(y_test, model.predict(X_test))
    print(f"degree {degree:2d}: train MSE = {train_mse:7.2f}   test MSE = {test_mse:7.2f}")



`np.linspace(start, stop, n)` — unlike `rng.uniform`, this produces `n` *evenly spaced*
values from `start` to `stop` (not random), which is handy for plotting smooth curves.

### Gradient descent, written from scratch

No new imports here — everything is built from numpy operations you've already used. The
function below implements the mechanics of gradient descent directly: start from a guess
(`w = np.zeros(...)`, all zeros), repeatedly compute the gradient of the squared-error
loss, and step a little in the opposite direction.


In [ ]:
def gradient_descent_linreg(X, y, lr=0.1, n_iter=200):
    n_samples, n_features = X.shape
    w = np.zeros(n_features)
    loss_history = []
    for _ in range(n_iter):
        y_pred = X @ w
        grad = -(2 / n_samples) * X.T @ (y - y_pred)
        w = w - lr * grad
        loss_history.append(np.mean((y - y_pred) ** 2))
    return w, loss_history

x_data = rng.uniform(0, 10, 60)
y_data = 2.5 * x_data + 3.0 + rng.normal(0, 4.0, 60)

x_std = (x_data - x_data.mean()) / x_data.std()   # standardizing helps convergence
X = np.column_stack([np.ones(60), x_std])

w, loss_history = gradient_descent_linreg(X, y_data, lr=0.3, n_iter=200)
print(f"final weights: {w}")
print(f"loss after 1 iteration: {loss_history[0]:.2f}, after 200: {loss_history[-1]:.2f}")



### Cross-validation

`cross_val_score` automates the "split into k folds, train/test k times, average the
scores" process. `scoring="neg_mean_squared_error"` is *negative* MSE because
scikit-learn's convention is that higher scores are always better — we negate it back
(`-scores`) to get an ordinary, positive MSE for printing.


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression

scores = cross_val_score(
    LinearRegression(), x.reshape(-1, 1), y, cv=5, scoring="neg_mean_squared_error"
)
print(f"per-fold MSE: {-scores}")
print(f"average MSE across folds: {-scores.mean():.3f}")



## Section 5 — Extending From Lines to Many-Dimensional Models

### Visualizing a loss surface as a grid of numbers

No new imports — just numpy again, used here to build a grid of loss values over a range
of possible `(b0, b1)` pairs. `[[... for b0 in b0_grid] for b1 in b1_grid]` is a **nested
list comprehension**: for every `b1`, compute the loss at every `b0`, producing a 2-D grid
that could be plotted as a surface (with matplotlib's 3D plotting, not shown here to keep
this cell fast).


In [ ]:
import numpy as np

rng = np.random.default_rng(42)
n = 60
x = rng.uniform(0, 10, n)
y = 2.5 * x + 3.0 + rng.normal(0, 4.0, n)

def loss_surface(b0, b1, x, y):
    return np.mean((y - (b0 + b1 * x)) ** 2)

b0_grid = np.linspace(-15, 20, 60)
b1_grid = np.linspace(-2, 6, 60)
loss_values = np.array([[loss_surface(b0, b1, x, y) for b0 in b0_grid] for b1 in b1_grid])

print(f"loss surface shape: {loss_values.shape}")
print(f"minimum loss on this grid: {loss_values.min():.2f}")



### A non-convex surface: Himmelblau's function

Plain Python functions this time (no sklearn or scipy needed) — `himmelblau` is the
landscape, `himmelblau_grad` is its gradient (worked out by hand with calculus), and
`gradient_descent` is a small, generic version of the optimizer from Section 4.


In [ ]:
def himmelblau(x, y):
    return (x**2 + y - 11) ** 2 + (x + y**2 - 7) ** 2

def himmelblau_grad(p):
    x, y = p
    dfdx = 4 * x * (x**2 + y - 11) + 2 * (x + y**2 - 7)
    dfdy = 2 * (x**2 + y - 11) + 4 * y * (x + y**2 - 7)
    return np.array([dfdx, dfdy])

def gradient_descent(grad_fn, start, lr=0.01, n_iter=300):
    p = np.array(start, dtype=float)
    for _ in range(n_iter):
        p = p - lr * grad_fn(p)
    return p

for start in [(-4, 4), (4, 4), (-4, -4), (4, -2)]:
    end_point = gradient_descent(himmelblau_grad, start)
    print(f"started at {start}, converged near {np.round(end_point, 2)}")



### A toy experiment: local minima get rarer as dimensions grow

`np.linalg.eigvalsh(...)` computes the eigenvalues of a symmetric matrix — used here as a
stand-in for checking whether a random point curves "upward" in every direction at once
(all eigenvalues positive means a true local minimum).


In [ ]:
rng = np.random.default_rng(0)

def fraction_true_local_minima(dimension, n_trials=300):
    count = 0
    for _ in range(n_trials):
        A = rng.normal(size=(dimension, dimension))
        hessian_stand_in = (A + A.T) / 2  # a random symmetric matrix, standing in for a Hessian
        eigenvalues = np.linalg.eigvalsh(hessian_stand_in)
        if np.all(eigenvalues > 0):        # positive in every direction => true local min
            count += 1
    return count / n_trials

for d in [2, 4, 8, 12]:
    frac = fraction_true_local_minima(d)
    print(f"dimension {d:2d}: fraction of critical points that are true local minima = {frac:.3f}")



### Feature scaling with `StandardScaler`

`StandardScaler` does exactly the z-score standardizing from Section 1 (`(x - mean) /
std`), but as a reusable scikit-learn object with a `.fit_transform()` method — handy once
you have many columns to standardize at once instead of one.


In [ ]:
from sklearn.preprocessing import StandardScaler

X_raw = np.column_stack([x_data, x_data * 1000])  # two features on very different scales
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
print(f"raw feature ranges: {X_raw.min(axis=0)} to {X_raw.max(axis=0)}")
print(f"scaled feature means: {X_scaled.mean(axis=0).round(3)}, stds: {X_scaled.std(axis=0).round(3)}")



## Section 6 — From Machine Learning to Artificial Intelligence

### A tiny neural network, built from scratch with numpy

No new libraries — a neural network's forward pass is just matrix multiplication
(`@`) and a simple nonlinear function (`relu`, which just zeroes out negative numbers)
applied in between two layers of weights.


In [ ]:
import numpy as np

def relu(z):
    return np.maximum(0, z)

def simple_forward_pass(x, W1, b1, W2, b2):
    '''
    x:  input vector
    W1, b1: weights and bias of the first (hidden) layer
    W2, b2: weights and bias of the second (output) layer
    '''
    hidden = relu(W1 @ x + b1)      # first layer: weighted sum, then nonlinearity
    output = W2 @ hidden + b2       # second layer: another weighted sum
    return output

rng = np.random.default_rng(0)
x_input = rng.normal(size=3)         # 3 input features
W1 = rng.normal(size=(4, 3)) * 0.5   # hidden layer: 4 hidden units
b1 = np.zeros(4)
W2 = rng.normal(size=(1, 4)) * 0.5   # output layer: 1 output value
b2 = np.zeros(1)

prediction = simple_forward_pass(x_input, W1, b1, W2, b2)
print(f"network output: {prediction}")



### Letting scikit-learn train a real (small) neural network

`MLPRegressor` ("Multi-Layer Perceptron") is scikit-learn's built-in neural network model.
It follows the exact same `.fit()` / `.predict()` interface as `LinearRegression` — that
consistency is deliberate on scikit-learn's part.

- `hidden_layer_sizes=(16, 16)` — two hidden layers, 16 neurons each.
- `activation="relu"` — the nonlinearity applied between layers.
- `max_iter=2000` — the maximum number of training iterations before giving up.


In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

x_data = np.linspace(-3, 3, 100).reshape(-1, 1)
y_data = np.sin(x_data).ravel() + rng.normal(0, 0.1, 100)

net = MLPRegressor(
    hidden_layer_sizes=(16, 16),   # two hidden layers, 16 units each
    activation="relu",
    max_iter=2000,
    random_state=0,
)
net.fit(x_data, y_data)
predictions = net.predict(x_data)

print(f"training MSE: {mean_squared_error(y_data, predictions):.4f}")
print(f"number of weight matrices: {len(net.coefs_)}")
print(f"total trainable parameters: {sum(w.size for w in net.coefs_) + sum(b.size for b in net.intercepts_)}")



(`.ravel()` flattens a 2-D array with one column back down to a plain 1-D array — needed
here because `x_data` is 2-D for scikit-learn's benefit, but `np.sin(x_data)` inherits that
same 2-D shape and we want a flat array of targets.)



## Section 7 — Using Python to Perform All of It

This section is a tour of the toolchain itself. If you haven't installed everything yet,
one command covers this whole module:

```bash
pip install numpy scipy scikit-learn matplotlib pandas
```

### NumPy: arrays and linear algebra, end to end

The complete normal-equations calculation from Section 2, in one place.


In [ ]:
import numpy as np

# The normal equations, end to end, using only NumPy
rng = np.random.default_rng(0)
x = rng.uniform(0, 10, 100)
y = 2.5 * x + 3.0 + rng.normal(0, 4.0, 100)

X = np.column_stack([np.ones_like(x), x])
beta = np.linalg.inv(X.T @ X) @ X.T @ y
print(f"beta = {beta}")



`np.ones_like(x)` creates an array of `1`s with the same shape as `x` — a small convenience
over writing `np.ones(len(x))` by hand.

### SciPy: an independent check


In [ ]:
from scipy import stats

result = stats.linregress(x, y)
print(f"slope = {result.slope:.3f}, intercept = {result.intercept:.3f}, r-squared = {result.rvalue**2:.3f}")



### scikit-learn: fit, then predict


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(
    x.reshape(-1, 1), y, test_size=0.3, random_state=0
)

model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print(f"test MSE = {mean_squared_error(y_test, predictions):.3f}")
print(f"test R^2 = {r2_score(y_test, predictions):.3f}")



### Matplotlib: seeing what the numbers mean

`import matplotlib.pyplot as plt` is the standard nickname — `plt` — for matplotlib's
main plotting interface. `plt.scatter` draws points, `plt.plot` draws a connected line,
and `plt.show()` displays the finished figure (in a Jupyter notebook, the figure usually
appears automatically even without `.show()`, but it's good practice to include it).


In [ ]:
import matplotlib.pyplot as plt

plt.scatter(x, y, alpha=0.7)
plt.plot(sorted(x), model.predict(np.sort(x).reshape(-1, 1)), color="crimson")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Data with fitted line")
plt.show()



### pandas: tabular data with labels

We reuse `x_data` / `y_data` from Section 6's neural network example here, to show moving
data between a DataFrame and the plain numpy arrays scikit-learn expects.


In [ ]:
import pandas as pd

df = pd.DataFrame({"x": x_data.ravel(), "y": y_data})
X_from_df = df[["x"]].to_numpy()   # extract as a NumPy array for sklearn
y_from_df = df["y"].to_numpy()

model = LinearRegression().fit(X_from_df, y_from_df)
print(f"fitted from a DataFrame: slope = {model.coef_[0]:.3f}")



### Reproducibility

The whole notebook relies on this pattern: fix a seed, and "random" numbers become
exactly repeatable.


In [ ]:
rng_a = np.random.default_rng(7)
rng_b = np.random.default_rng(7)
print(np.array_equal(rng_a.normal(size=5), rng_b.normal(size=5)))  # True: identical seeds, identical draws



### Quick reference: concept → library

| Concept | Section | Library / function |
|---|---|---|
| Mean, variance, covariance, correlation | 1 | `numpy` |
| Correlation matrix across many variables | 1 | `pandas.DataFrame.corr()` |
| Normal equations (by hand) | 2 | `numpy.linalg.inv` |
| Least-squares fit (verified) | 2 | `scipy.stats.linregress` |
| Multiple / polynomial regression | 2 | `sklearn.linear_model.LinearRegression`, `PolynomialFeatures` |
| MSE, MAE, R² | 3 | `sklearn.metrics` |
| Train/test split, cross-validation | 4 | `sklearn.model_selection` |
| Gradient descent (from scratch) | 4 | `numpy` |
| Loss surfaces, non-convex optimization | 5 | `numpy` + `matplotlib` (3D plotting) |
| Feature scaling | 5 | `sklearn.preprocessing.StandardScaler` |
| All plots and figures | 1–5 | `matplotlib` |



## Section 8 — From Fitting Lines to Recognizing a Cat in a Photo

### Step 1 — a photo is just a large table of numbers

`sklearn.datasets.load_digits()` ships a small, built-in data set of handwritten digit
images directly inside scikit-learn — no download and no internet connection required,
which is exactly why we use it here instead of real cat photographs. Each image is an 8x8
grid of numbers (pixel brightness values), stored as a numpy array.


In [ ]:
import numpy as np
from sklearn.datasets import load_digits

digits = load_digits()
image = digits.images[0]         # an 8x8 grayscale image, as a NumPy array
print(image.shape)                 # (8, 8)
print(image)                        # each entry is a pixel brightness value



### Step 2 — treat every pixel as a predictor, and classification as fitting

`image.reshape(len(images), -1)` **flattens** each 8x8 grid into a single row of 64
numbers — same idea as flattening a table, just applied to every image in the data set at
once. `digits.target` holds the true digit (0-9) for each image — the label we're trying
to predict.

`LogisticRegression` is scikit-learn's model for classification (predicting a category
instead of a number) — same `.fit()` / `.predict()` interface you've already used several
times. `accuracy_score` reports the fraction of test images the model classified
correctly.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = digits.images.reshape(len(digits.images), -1)   # flatten each 8x8 image to 64 numbers
y = digits.target                                     # the true digit, 0-9, for each image

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0
)

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)
predictions = clf.predict(X_test)

print(f"test accuracy: {accuracy_score(y_test, predictions):.3f}")
print(f"number of weights being fit: {clf.coef_.size}")



### Step 4 — a hand-designed convolutional kernel

No new libraries here — this is a from-scratch, plain-numpy implementation of a single
2-D convolution, to make the mechanics concrete before you meet a real deep-learning
library's (much faster, GPU-accelerated) version of the same operation.


In [ ]:
def apply_kernel(image, kernel):
    '''A minimal, from-scratch 2D convolution -- no padding, unit stride.'''
    kh, kw = kernel.shape
    ih, iw = image.shape
    out_h, out_w = ih - kh + 1, iw - kw + 1
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            patch = image[i:i + kh, j:j + kw]
            output[i, j] = np.sum(patch * kernel)   # the same weighted-sum idea, applied locally
    return output

vertical_edge_kernel = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1],
])

sample_image = digits.images[0]
edges = apply_kernel(sample_image, vertical_edge_kernel)
print(f"original shape: {sample_image.shape}, after kernel: {edges.shape}")
print(np.round(edges, 1))



---

## You made it through the whole stack

Every cell in this notebook builds on the ones before it: `numpy` arrays hold the
numbers, `scipy` and `sklearn` fit and score models on them, `pandas` organizes them into
tables, and `matplotlib` lets you look at the result. That combination — and the habit of
fitting a model, measuring its error, and checking it on data it hasn't seen — is the same
loop underneath everything from a two-parameter regression to a full neural network.
